# iTransformer trên PTB-XL: train từ đầu + MIA cơ bản (TV3)

Pipeline: tải PTB-XL → split member/non-member → train **iTransformer** (classification) → Loss / Confidence / Entropy → TPR @ 0,1% & 1% FPR + bootstrap CI → ghi `results/itransformer/cls/{loss,conf,entropy}.json`.

**Trước khi chạy**
1. Runtime → GPU (A100/L4).
2. Upload lên Drive `MyDrive/Final_Project/`:
   - `itransformer/`: thư mục `experiment/itransformer` (có `setup.py`)
   - `dataset/`: có `make_split.py`, `splits/`, `archive.zip`
3. Chạy tuần tự. Checkpoint resume trong `ckpt/itransformer_cls`.


In [ ]:
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/Final_Project'
ITX_SRC = f'{DRIVE_ROOT}/itransformer'      # folder containing setup.py
DATASET_DIR = f'{DRIVE_ROOT}/dataset'
PTBXL_ZIP = f'{DATASET_DIR}/archive.zip'
RESULTS_DIR = f'{DRIVE_ROOT}/results/itransformer/cls'

!nvidia-smi --query-gpu=name,memory.total --format=csv


In [ ]:
!rm -rf /content/itransformer && cp -r {ITX_SRC} /content/itransformer
!pip install -q /content/itransformer wfdb scikit-learn pandas tqdm matplotlib


## 1. Cấu hình
Train từ đầu: epoch/lr tự chọn theo utility trên `val` (tối đa 6 cấu hình). Recipe mặc định dưới đây là điểm xuất phát.


In [ ]:
SEED = 42
LEVEL = 'record'
EPOCHS = 50
EVAL_AT = [5, 10, 25, 50]     # 10%, 20%, 50%, 100% of 50
BATCH = 16
MAX_LR = 1e-3
N_VAL = 500
N_BOOT = 1000
CKPT_EVERY = 1
CKPT_DIR = f'{DRIVE_ROOT}/ckpt/itransformer_cls'

# iTransformer hyperparams
D_MODEL, N_HEADS, E_LAYERS, D_FF, DROPOUT = 128, 8, 3, 256, 0.1
SEQ_LEN = 512

PTBXL_ROOT = '/content/ptbxl_raw'
SPLIT_PATH = f'{DATASET_DIR}/splits/ptbxl_{LEVEL}_s{SEED}.json'
CACHE_DIR = f'/content/cache/itx_cls_{LEVEL}_s{SEED}'


## 2. Dữ liệu PTB-XL


In [ ]:
import os, glob
if not glob.glob(f'{PTBXL_ROOT}/**/ptbxl_database.csv', recursive=True):
    os.makedirs(PTBXL_ROOT, exist_ok=True)
    assert os.path.exists(PTBXL_ZIP), f'missing {PTBXL_ZIP}; {DATASET_DIR} has {os.listdir(DATASET_DIR)}'
    !unzip -q -o "{PTBXL_ZIP}" -d {PTBXL_ROOT}
found = glob.glob(f'{PTBXL_ROOT}/**/ptbxl_database.csv', recursive=True)
assert found, f'no ptbxl_database.csv after unzip; {PTBXL_ROOT} has {os.listdir(PTBXL_ROOT)}'
PTBXL_DIR = os.path.dirname(found[0])
print(PTBXL_DIR, os.listdir(PTBXL_DIR))


In [ ]:
import json, random, subprocess, sys, os
import numpy as np, torch, pandas as pd
from itransformer.ptbxl import PTBXLDataset, load_split_meta

def control_randomness(seed=SEED):
    random.seed(seed); os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
control_randomness()

if not os.path.exists(SPLIT_PATH):
    subprocess.run([sys.executable, f'{DATASET_DIR}/make_split.py', '--ptbxl-dir', PTBXL_DIR,
                    '--level', LEVEL, '--seed', str(SEED), '--n-val', str(N_VAL)], check=True)
split, meta = load_split_meta(SPLIT_PATH)
print({k: len(v) for k, v in split.items()}, 'sha256', meta['sha256'][:12])
if len(split['test']) < 10000:
    print('WARNING: fewer than 10k non-members; TPR@0.1%FPR rests on', int(0.001 * len(split['test'])), 'samples')

member_ds = PTBXLDataset(PTBXL_DIR, SPLIT_PATH, 'train', SEQ_LEN, CACHE_DIR)
val_ds    = PTBXLDataset(PTBXL_DIR, SPLIT_PATH, 'val',   SEQ_LEN, CACHE_DIR)
nonmem_ds = PTBXLDataset(PTBXL_DIR, SPLIT_PATH, 'test',  SEQ_LEN, CACHE_DIR)
print(len(member_ds), len(val_ds), len(nonmem_ds), member_ds.timeseries.shape)


## 3. Train + tấn công
Score: **cao hơn = nhiều khả năng là member**. Loss: `-CE`, Confidence: `max softmax`, Entropy: `-H(p)`.


In [ ]:
import time
import torch.nn.functional as F
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from itransformer.model import ITransformer
from itransformer.mia import attack_scores_cls, mia_metrics, bootstrap_ci

dev = 'cuda' if torch.cuda.is_available() else 'cpu'
bf16 = torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8
amp = lambda: torch.autocast(device_type='cuda', dtype=torch.bfloat16, enabled=bf16)

def build_model():
    model = ITransformer(
        task='classification', seq_len=SEQ_LEN, n_variates=12, n_class=5,
        d_model=D_MODEL, n_heads=N_HEADS, e_layers=E_LAYERS, d_ff=D_FF, dropout=DROPOUT,
    )
    n_tr = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'trainable {n_tr/1e6:.2f}M')
    return model.to(dev)

@torch.no_grad()
def get_logits(model, ds, bs=64):
    model.eval(); out = []
    for x, _ in DataLoader(ds, batch_size=bs, shuffle=False):
        with amp():
            out.append(model(x.to(dev).float()).float().cpu())
    return torch.cat(out)

@torch.no_grad()
def accuracy(model, ds, bs=64):
    logits = get_logits(model, ds, bs)
    return float((logits.argmax(1).numpy() == ds.labels).mean()), logits

def save_atomic(obj, path):
    tmp = path + '.tmp'
    torch.save(obj, tmp); os.replace(tmp, path)
    with open(path, 'rb') as _f: os.fsync(_f.fileno())


In [ ]:
def run_train():
    os.makedirs(CKPT_DIR, exist_ok=True)
    tag = f'itx_{LEVEL}_s{SEED}'
    ckpt_path, done_path = f'{CKPT_DIR}/{tag}.pt', f'{CKPT_DIR}/{tag}.done.json'
    key = f"itx|cls|{LEVEL}|{SEED}|{EPOCHS}|{BATCH}|{MAX_LR}|{D_MODEL}|{meta['sha256'][:12]}"
    if os.path.exists(done_path) and json.load(open(done_path)).get('key') == key:
        print('already finished, skipping'); return json.load(open(done_path))['history']

    control_randomness()
    model = build_model()
    loader = DataLoader(member_ds, batch_size=BATCH, shuffle=True, drop_last=True)
    params = list(model.parameters())
    opt = torch.optim.AdamW(params, lr=MAX_LR / 25)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=MAX_LR, total_steps=EPOCHS * len(loader))
    ce = torch.nn.CrossEntropyLoss()
    history, start = [], 1

    if os.path.exists(ckpt_path):
        ck = torch.load(ckpt_path, map_location='cpu')
        if ck.get('key') == key:
            model.load_state_dict(ck['model']); opt.load_state_dict(ck['opt']); sched.load_state_dict(ck['sched'])
            history, start = ck['history'], ck['epoch'] + 1
            print(f'resumed after epoch {ck["epoch"]}')
        del ck

    for ep in range(start, EPOCHS + 1):
        model.train(); losses = []
        for x, yb in tqdm(loader, desc=f'ep{ep}/{EPOCHS}', leave=False):
            opt.zero_grad(set_to_none=True)
            with amp():
                logits = model(x.to(dev).float())
            loss = ce(logits.float(), yb.to(dev))
            loss.backward()
            torch.nn.utils.clip_grad_norm_(params, 1.0)
            opt.step(); sched.step(); losses.append(loss.item())

        if ep in EVAL_AT:
            acc_m, lm = accuracy(model, member_ds)
            acc_n, ln = accuracy(model, nonmem_ds)
            # Also report val utility for hyperparam selection / Báo cáo utility val để chọn HP
            acc_v, _ = accuracy(model, val_ds)
            save_atomic({'lm': lm, 'ln': ln}, f'{CKPT_DIR}/{tag}_logits_ep{ep}.pt')
            sm, sn = attack_scores_cls(lm, member_ds.labels), attack_scores_cls(ln, nonmem_ds.labels)
            row = {'epoch': ep, 'train_loss': float(np.mean(losses)),
                   'train_acc': acc_m, 'val_acc': acc_v, 'test_acc': acc_n, 'gap': acc_m - acc_n}
            for a in sm:
                t01, t1, au = mia_metrics(sm[a], sn[a])
                row.update({f'{a}_tpr@0.1': t01, f'{a}_tpr@1': t1, f'{a}_auc': au})
            history.append(row)
            print({k: (round(v, 4) if isinstance(v, float) else v) for k, v in row.items()})

        if ep % CKPT_EVERY == 0 or ep == EPOCHS or ep in EVAL_AT:
            save_atomic({'key': key, 'epoch': ep, 'history': history,
                         'opt': opt.state_dict(), 'sched': sched.state_dict(),
                         'model': {k: v.cpu() for k, v in model.state_dict().items()}}, ckpt_path)

    final_ep = max(e for e in EVAL_AT if e <= EPOCHS)
    row = [r for r in history if r['epoch'] == final_ep][-1]
    lg = torch.load(f'{CKPT_DIR}/{tag}_logits_ep{final_ep}.pt')
    sm = attack_scores_cls(lg['lm'], member_ds.labels)
    sn = attack_scores_cls(lg['ln'], nonmem_ds.labels)
    os.makedirs(RESULTS_DIR, exist_ok=True)
    for a in sm:
        c01, c1 = bootstrap_ci(sm[a], sn[a], N_BOOT, SEED)
        out = {
            'model': 'itransformer', 'task': 'cls', 'attack': a,
            'owner_tv': 3, 'level': LEVEL, 'dataset': 'ptbxl', 'seed': SEED,
            'n_member': int(len(sm[a])), 'n_nonmember': int(len(sn[a])),
            'scores_member': np.round(sm[a], 6).tolist(),
            'scores_nonmember': np.round(sn[a], 6).tolist(),
            'tpr_at_0.1fpr': row[f'{a}_tpr@0.1'], 'tpr_at_1fpr': row[f'{a}_tpr@1'], 'auc': row[f'{a}_auc'],
            'bootstrap_ci': {'n_resamples': N_BOOT, 'confidence': 0.95, 'tpr_at_0.1fpr': c01, 'tpr_at_1fpr': c1},
            'utility': {'train_acc': row['train_acc'], 'test_acc': row['test_acc'], 'gap': row['gap']},
            'n_shadow': None,
            'extra': {'eval_epochs': EVAL_AT, 'tuning': 'default recipe; tune on val_acc only',
                      'epochs': EPOCHS, 'eval_epoch': row['epoch'], 'max_lr': MAX_LR, 'batch': BATCH,
                      'backbone': 'iTransformer', 'd_model': D_MODEL, 'n_heads': N_HEADS,
                      'e_layers': E_LAYERS, 'd_ff': D_FF, 'val_acc': row.get('val_acc')},
        }
        fn = f'{RESULTS_DIR}/{a}.json'
        json.dump(out, open(fn, 'w')); print('saved', fn)
    json.dump({'key': key, 'history': history}, open(done_path, 'w'))
    del model, opt
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return history

hist = run_train()
pd.DataFrame(hist)


## 4. ROC log–log (Loss, epoch cuối)


In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve
fig, ax = plt.subplots(figsize=(5, 5))
for a in ['loss', 'conf', 'entropy']:
    r = json.load(open(f'{RESULTS_DIR}/{a}.json'))
    sm, sn = np.array(r['scores_member']), np.array(r['scores_nonmember'])
    fpr, tpr, _ = roc_curve(np.r_[np.ones(len(sm)), np.zeros(len(sn))], np.r_[sm, sn])
    ax.plot(fpr, tpr, label=f"{a} AUC={r['auc']:.3f}")
ax.plot([1e-4, 1], [1e-4, 1], 'k--', lw=0.8)
ax.set_xscale('log'); ax.set_yscale('log'); ax.set_xlim(1e-4, 1); ax.set_ylim(1e-4, 1)
ax.set_xlabel('FPR'); ax.set_ylabel('TPR'); ax.legend(); ax.set_title('iTransformer cls')
plt.show()
